In [184]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
import gradio as gr
import uuid
import asyncio
import os
from datetime import datetime
from dotenv import load_dotenv
import aiosqlite
from langchain.agents import Tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_experimental.tools import PythonREPLTool
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

In [185]:
load_dotenv(override=True)

True

In [186]:
# SQL Memory Class (Persistent Memory with Thread Support)

class SQLMemory:
    def __init__(self, db_path="memory.db"):
        self.db_path = db_path

    async def init_db(self):
        async with aiosqlite.connect(self.db_path) as db:
            await db.execute("""
                CREATE TABLE IF NOT EXISTS conversations (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    thread_id TEXT,
                    role TEXT,
                    content TEXT,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
                )
            """)
            await db.commit()

    async def save_message(self, thread_id: str, role: str, content: str):
        async with aiosqlite.connect(self.db_path) as db:
            await db.execute(
                "INSERT INTO conversations (thread_id, role, content) VALUES (?, ?, ?)",
                (thread_id, role, content)
            )
            await db.commit()

    async def load_messages(self, thread_id: str):
        async with aiosqlite.connect(self.db_path) as db:
            cursor = await db.execute(
                "SELECT role, content FROM conversations WHERE thread_id = ? ORDER BY id ASC",
                (thread_id,)
            )
            rows = await cursor.fetchall()

        messages = []
        for role, content in rows:
            if role == "user":
                messages.append(HumanMessage(content=content))
            elif role == "assistant":
                messages.append(AIMessage(content=content))
            elif role == "system":
                messages.append(SystemMessage(content=content))

        return messages

    async def get_memory_context(self, thread_id: str, limit: int = 10) -> str:
        """
        Returns last N messages as formatted text for prompt injection
        """
        async with aiosqlite.connect(self.db_path) as db:
            cursor = await db.execute(
                "SELECT role, content FROM conversations WHERE thread_id = ? ORDER BY id DESC LIMIT ?",
                (thread_id, limit)
            )
            rows = await cursor.fetchall()

        # Reverse to maintain chronological order
        rows = rows[::-1]

        context = "Previous conversation:\n"
        for role, content in rows:
            context += f"{role.upper()}: {content}\n"

        return context

In [187]:
# State Definition (Clean Final)

class State(TypedDict):
    messages: Annotated[List[Any], add_messages]

    thread_id: str
    memory_context: Optional[str]

    success_criteria: str

    plan: Optional[List[str]]
    current_step: Optional[int]

    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

In [188]:
# Memory Initialization + Thread ID Setup

memory = SQLMemory()

async def setup_memory():
    await memory.init_db()

def generate_thread_id(username: str):
    session_id = str(uuid.uuid4())[:8]
    return f"{username}_{session_id}"

In [189]:
# Tools Setup

KEY = os.getenv("SERPER_API_KEY")

if not KEY:
    print("No SERPER_API_KEY found")
    
serper = GoogleSerperAPIWrapper(serper_api_key=os.getenv("SERPER_API_KEY"))

def get_tools():
    search_tool = Tool(
        name="search",
        func=serper.run,
        description="Use this tool to search the web for information"
    )

    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    python_tool = PythonREPLTool()

    return [search_tool, wiki, python_tool]

In [190]:
# Clarifier Agent

llm = ChatOpenAI(model="gpt-4o-mini")

def clarifier(state: State) -> Dict[str, Any]:
    system_message = f"""
You are a clarification assistant.

Your job is to ask 2 to 3 clear, specific questions that will help better understand the user's request before solving it.

Do not attempt to solve the task.
Only ask questions.

User request:
{state["messages"][-1].content}
"""

    response = llm.invoke([SystemMessage(content=system_message)])

    return {
        "messages": [response],
        "user_input_needed": True
    }

In [191]:
# Planner Agent

def planner(state: State) -> Dict[str, Any]:
    user_request = state["messages"][0].content
    clarifications = state.get("clarifications", "")

    system_message = f"""
You are a planning assistant.

Your job is to break down the user's task into a clear sequence of steps.

Use the user's request and any clarifications provided.

Return a numbered list of steps.

User request:
{user_request}

Clarifications:
{clarifications}
"""

    response = llm.invoke([SystemMessage(content=system_message)])

    plan_text = response.content

    steps = [step.strip() for step in plan_text.split("\n") if step.strip()]

    return {
        "messages": [response],
        "plan": steps,
        "current_step": 0
    }

In [192]:
# Worker Agent (Adaptive)

tools = get_tools()
llm_with_tools = llm.bind_tools(tools)

def worker(state: State) -> Dict[str, Any]:
    plan = state.get("plan", [])
    step_index = state.get("current_step", 0)
    memory_context = state.get("memory_context", "")

    current_task = ""
    if plan and step_index < len(plan):
        current_task = plan[step_index]

    system_message = f"""
You are an intelligent assistant that can use tools.

Your goal is to complete the user's request step by step.

You may:
- Use tools to gather information
- Ask the user a question if something is unclear
- Continue working until the task is complete

If you ask a question, clearly prefix it with:
Question:

If the task is complete, provide the final answer.

Success criteria:
{state["success_criteria"]}

Current step:
{current_task}

Previous context:
{memory_context}
"""

    if state.get("feedback_on_work"):
        system_message += f"""
Your previous answer did not meet the success criteria.

Feedback:
{state["feedback_on_work"]}

Improve your answer.
"""

    messages = [SystemMessage(content=system_message)] + state["messages"]

    response = llm_with_tools.invoke(messages)

    return {
        "messages": [response],
        "current_step": step_index + 1
    }

In [193]:
# Evaluator Agent

class EvaluatorOutput(BaseModel):
    feedback: str
    success_criteria_met: bool
    user_input_needed: bool


evaluator_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EvaluatorOutput)

def format_conversation(messages: List[Any]) -> str:
    conversation = ""
    for m in messages:
        if isinstance(m, HumanMessage):
            conversation += f"User: {m.content}\n"
        elif isinstance(m, AIMessage):
            conversation += f"Assistant: {m.content}\n"
    return conversation


def evaluator(state: State) -> Dict[str, Any]:
    last_response = state["messages"][-1].content
    success_criteria = state["success_criteria"]

    system_message = """
You are an evaluator.

Determine if the assistant has successfully completed the task.

Return:
- feedback
- whether success criteria is met
- whether user input is needed
"""

    user_message = f"""
Conversation:
{format_conversation(state["messages"])}

Success criteria:
{success_criteria}

Last response:
{last_response}
"""

    result = evaluator_llm.invoke([
        SystemMessage(content=system_message),
        HumanMessage(content=user_message)
    ])

    return {
        "messages": [AIMessage(content=f"Evaluator: {result.feedback}")],
        "feedback_on_work": result.feedback,
        "success_criteria_met": result.success_criteria_met,
        "user_input_needed": result.user_input_needed
    }

In [194]:
# Graph Construction (Final Fixed)

def build_graph():
    graph_builder = StateGraph(State)

    graph_builder.add_node("planner", planner)
    graph_builder.add_node("worker", worker)
    graph_builder.add_node("tools", ToolNode(tools=tools))
    graph_builder.add_node("evaluator", evaluator)

    def worker_router(state: State):
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        return "evaluator"

    def evaluator_router(state: State):
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        return "worker"

    graph_builder.add_edge(START, "planner")
    graph_builder.add_edge("planner", "worker")
    graph_builder.add_edge("tools", "worker")

    graph_builder.add_conditional_edges("worker", worker_router, {
        "tools": "tools",
        "evaluator": "evaluator"
    })

    graph_builder.add_conditional_edges("evaluator", evaluator_router, {
        "worker": "worker",
        "END": END
    })

    return graph_builder.compile()

In [195]:
# Run Logic (Streaming Clean Output)

graph = build_graph()

initialized = False

async def run_agent(username: str, message: str, success_criteria: str, thread_id: str):
    global initialized

    if not initialized:
        await memory.init_db()
        initialized = True

    past_messages = await memory.load_messages(thread_id)
    memory_context = await memory.get_memory_context(thread_id)

    current_messages = past_messages + [HumanMessage(content=message)]

    state = {
        "messages": current_messages,
        "thread_id": thread_id,
        "memory_context": memory_context,
        "success_criteria": "Provide a complete, accurate, and actionable response that fully addresses the user's request",
        "plan": None,
        "current_step": 0,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False
    }

    final_response = ""

    async for event in graph.astream(state):
        for node, value in event.items():

            if "messages" in value:
                msg = value["messages"][-1]

                if isinstance(msg, AIMessage):
                    content = msg.content.strip()

                    if content.lower().startswith("evaluator:"):
                        continue

                    final_response = content

                    yield (node, content)

    await memory.save_message(thread_id, "user", message)
    await memory.save_message(thread_id, "assistant", final_response)

In [196]:
# Authentication System (Final Safe Version)

import hashlib

def hash_password(password: str):
    return hashlib.sha256(password.encode()).hexdigest()


async def create_users_table():
    async with aiosqlite.connect("memory.db") as db:
        await db.execute("""
            CREATE TABLE IF NOT EXISTS users (
                username TEXT PRIMARY KEY,
                password TEXT
            )
        """)
        await db.commit()


async def register_user(username: str, password: str):
    await create_users_table()

    hashed = hash_password(password)

    async with aiosqlite.connect("memory.db") as db:
        try:
            await db.execute(
                "INSERT INTO users (username, password) VALUES (?, ?)",
                (username, hashed)
            )
            await db.commit()
            return "User registered successfully"
        except:
            return "Username already exists"


async def login_user(username: str, password: str):
    await create_users_table()

    hashed = hash_password(password)

    async with aiosqlite.connect("memory.db") as db:
        cursor = await db.execute(
            "SELECT password FROM users WHERE username = ?",
            (username,)
        )
        row = await cursor.fetchone()

    if row and row[0] == hashed:
        return True
    return False

In [197]:
# Gradio UI (Clean UX + Clear Button)

import gradio as gr

def format_status(node, content):
    if node == "planner":
        return "Planning..."
    elif node == "worker":
        if "search" in content.lower():
            return "Searching..."
        return "Thinking..."
    elif node == "evaluator":
        return "Finalizing..."
    return "Processing..."


def clean_text(text):
    if ":" in text:
        parts = text.split(":", 1)
        if parts[0].lower() in ["worker", "planner", "evaluator"]:
            return parts[1].strip()
    return text


async def login(username, password):
    success = await login_user(username, password)

    if success:
        thread_id = generate_thread_id(username)
        return (
            username,
            thread_id,
            gr.update(visible=True),
            gr.update(visible=False),
            "Login successful"
        )
    else:
        return (
            None,
            None,
            gr.update(visible=False),
            gr.update(visible=True),
            "Invalid credentials"
        )


async def register(username, password):
    result = await register_user(username, password)
    return result


async def chat(username, message, thread_id, history):
    history = history + [(message, "Processing...")]

    yield history, thread_id, ""

    final_answer = ""

    async for node, content in run_agent(username, message, "", thread_id):
        content = clean_text(content)

        status = format_status(node, content)
        history[-1] = (message, status)
        yield history, thread_id, ""

        final_answer = content

    history[-1] = (message, final_answer)
    yield history, thread_id, ""


def clear_chat():
    return [], []


with gr.Blocks() as demo:
    gr.Markdown("## AI Sidekick Agent")

    user_state = gr.State()
    thread_state = gr.State()
    chat_history = gr.State([])

    # AUTH UI
    with gr.Group(visible=True) as auth_ui:
        username_input = gr.Textbox(label="Username")
        password_input = gr.Textbox(label="Password", type="password")

        login_btn = gr.Button("Login")
        register_btn = gr.Button("Register")

        auth_status = gr.Textbox(label="Status")

    # CHAT UI
    with gr.Group(visible=False) as chat_ui:
        chatbot = gr.Chatbot()
        message = gr.Textbox(label="Message")

        with gr.Row():
            send_btn = gr.Button("Send")
            clear_btn = gr.Button("Clear")

    login_btn.click(
        login,
        inputs=[username_input, password_input],
        outputs=[user_state, thread_state, chat_ui, auth_ui, auth_status]
    )

    register_btn.click(
        register,
        inputs=[username_input, password_input],
        outputs=[auth_status]
    )

    send_btn.click(
        chat,
        inputs=[user_state, message, thread_state, chat_history],
        outputs=[chatbot, thread_state, message]
    )

    clear_btn.click(
        clear_chat,
        inputs=[],
        outputs=[chatbot, chat_history]
    )

demo.launch()

/tmp/ipykernel_125114/2992183099.py:95: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
